In [1]:
import json, random
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import os

import wandb

# ─────────────────────────────────────────────────────────────────────────────
# 재현성을 위한 시드 고정
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ─────────────────────────────────────────────────────────────────────────────

# 파일 저장 폴더
SAVE_DIR = "./saved_bert4rec"
os.makedirs(SAVE_DIR, exist_ok=True)

# ======================================================
# 1) 데이터 로드 & user_seqs 생성 (변경 없음)
# ======================================================
with open('./util/result_clean.json','r') as f:
    raw_data = json.load(f)

rows = []
for uid, user in enumerate(raw_data):
    for ts, token in enumerate(user['token_sequence']):
        rows.append([uid, token, ts])
df = pd.DataFrame(rows, columns=["user_id","item_id","timestamp"])
user_seqs = df.groupby("user_id")["item_id"].apply(list).tolist()

# ======================================================
# 2) 토큰 ↔ ID 매핑 (변경 없음)
# ======================================================
unique_items = sorted(df["item_id"].unique().tolist())
token2id = {t:i+1 for i,t in enumerate(unique_items)}
token2id['[MASK]'] = len(token2id) + 1
id2token = {v:k for k,v in token2id.items()}

# JSON으로 저장
with open(os.path.join(SAVE_DIR, "token2id.json"), "w", encoding="utf-8") as f:
    json.dump(token2id, f, ensure_ascii=False, indent=2)
with open(os.path.join(SAVE_DIR, "id2token.json"), "w", encoding="utf-8") as f:
    json.dump(id2token, f, ensure_ascii=False, indent=2)

# ======================================================
# 3) Leave-two-out 방식으로 분할:
#      - train_seqs: seq[:-2]
#      - val_seqs:   seq[:-1]
#      - test_seqs:  seq (원본 전체, 마지막 아이템이 테스트 대상)
# ======================================================
train_seqs = []
val_seqs   = []
test_seqs  = []

for seq in user_seqs:
    # 길이가 2 이하인 경우는 skip (train/val/test가 모두 성립하려면 최소 3개 이상)
    if len(seq) < 3:
        continue

    # train: 마지막 두 개 아이템 제외
    train_seqs.append(seq[:-2])

    # validation: 마지막 아이템 한 개만 제외 → BERT4RecDataset(val=True)에서
    #             마지막 non-pad 위치(=원본 두 번째 마지막)를 마스킹하여 평가
    val_seqs.append(seq[:-1])

    # test: 원본 전체 시퀀스 → BERT4RecDataset(val=True)에서
    #       마지막 non-pad 위치(=원본 마지막)를 마스킹하여 평가
    test_seqs.append(seq)

# ======================================================
# 4) Dataset & DataLoader 정의
# ======================================================
class BERT4RecDataset(Dataset):
    def __init__(self, sequences, token2id, max_len=20, mask_ratio=0.2, val=False):
        self.sequences     = sequences
        self.token2id      = token2id
        self.max_len       = max_len
        self.mask_ratio    = mask_ratio
        self.val           = val
        self.mask_token_id = token2id['[MASK]']

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        # 1) 토큰→ID, truncate, left-pad
        ids = [self.token2id[t] for t in seq if t in self.token2id]
        ids = ids[-self.max_len:]
        pad = [0] * (self.max_len - len(ids))
        input_ids = pad + ids

        labels     = [-100] * self.max_len
        masked_ids = input_ids.copy()

        if self.val:
            # validation/Test: 마지막 non-pad 토큰만 마스크
            last_pos = max(i for i, x in enumerate(input_ids) if x != 0)
            labels[last_pos]     = input_ids[last_pos]
            masked_ids[last_pos] = self.mask_token_id
        else:
            # train: 랜덤 마스크 + 최소 1개 보장
            for i in range(self.max_len):
                if masked_ids[i] != 0 and random.random() < self.mask_ratio:
                    labels[i]       = masked_ids[i]
                    masked_ids[i]   = self.mask_token_id
            if all(l == -100 for l in labels):
                # 만약 모든 위치에서 마스킹이 일어나지 않았다면, 거듭 확인하여 
                # 마지막 non-pad 위치는 무조건 한 번 마스킹 처리
                last_pos = max(i for i,x in enumerate(input_ids) if x != 0)
                labels[last_pos]     = input_ids[last_pos]
                masked_ids[last_pos] = self.mask_token_id

        return (
            torch.tensor(masked_ids, dtype=torch.long),
            torch.tensor(labels,     dtype=torch.long),
        )

config = {
    "n_layers": 4,
    "n_heads": 4,
    "hidden_size": 64,
    "inner_size": 256,
    "hidden_dropout_prob": 0.2,
    "attn_dropout_prob": 0.2,
    "hidden_act": "gelu",
    "layer_norm_eps": 1e-12,
    "initializer_range": 0.02,
    "mask_ratio": 0.2,
    "loss_type": "CE",
    "max_seq_length": 20,
    "n_items": len(token2id)
}

wandb.init(
    project="seq_rec",
    entity="ai_project_team2",
    name="bert4rec_leave2out",
    config=config
)

BATCH_SIZE = 128

# ─ Train DataLoader ─
train_ds = BERT4RecDataset(
    train_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=config['mask_ratio'],  # train 단계: 랜덤 마스크
    val=False
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# ─ Validation DataLoader ─
val_ds = BERT4RecDataset(
    val_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,  # val 단계: mask_ratio=0.0으로 설정
    val=True         # val=True → 마지막 non-pad 위치(=원본 seq의 penultimate)를 마스킹
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ─ Test DataLoader ─
test_ds = BERT4RecDataset(
    test_seqs,
    token2id,
    max_len=config['max_seq_length'],
    mask_ratio=0.0,  # test 단계: mask_ratio=0.0
    val=True         # val=True → 마지막 non-pad 위치(=원본 seq의 마지막)를 마스킹
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# ======================================================
# 5) BERT4Rec 모델 정의 (이전과 동일)
# ======================================================
class BERT4Rec(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.hidden_size = cfg['hidden_size']
        self.max_len     = cfg['max_seq_length']
        self.n_items     = cfg['n_items']

        # +2: padding(0), mask 토큰
        self.item_emb = nn.Embedding(self.n_items+2, self.hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(self.max_len, self.hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_size, nhead=cfg['n_heads'],
            dim_feedforward=cfg['inner_size'], dropout=cfg['hidden_dropout_prob'],
            activation="gelu", layer_norm_eps=cfg['layer_norm_eps'],
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['n_layers'])
        self.norm    = nn.LayerNorm(self.hidden_size, eps=cfg['layer_norm_eps'])
        self.drop    = nn.Dropout(cfg['hidden_dropout_prob'])
        self.out     = nn.Linear(self.hidden_size, self.n_items+1)
        self._init_weights(cfg['initializer_range'])

    def _init_weights(self, std):
        for n,p in self.named_parameters():
            if 'weight' in n:
                nn.init.normal_(p, 0, std)
            elif 'bias' in n:
                nn.init.constant_(p, 0)

    def forward(self, input_ids):
        pos = torch.arange(self.max_len, device=input_ids.device).unsqueeze(0).expand_as(input_ids)
        x = self.item_emb(input_ids) + self.pos_emb(pos)
        x = self.norm(x)
        x = self.drop(x)

        pad_mask = (input_ids == 0)
        h = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.out(h)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert = BERT4Rec({**config, **{"n_items": len(token2id)}}).to(device)

opt       = torch.optim.Adam(model_bert.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# ======================================================
# 6) Train + Validation 루프
#    - Validation Loss가 가장 낮을 때 모델 저장
# ======================================================
best_val_loss = float('inf')
best_epoch    = -1
EPOCHS = 100


def evaluate_simple_metrics(all_scores: np.ndarray, all_labels: np.ndarray):
    N, V = all_scores.shape
    rank = np.argsort(-all_scores, axis=1)

    hits1  = np.array([1 if all_labels[i] in rank[i, :1] else 0 for i in range(N)])
    hits5  = np.array([1 if all_labels[i] in rank[i, :5] else 0 for i in range(N)])
    hits10 = np.array([1 if all_labels[i] in rank[i, :10] else 0 for i in range(N)])
    HR1  = hits1.mean()
    HR5  = hits5.mean()
    HR10 = hits10.mean()

    def compute_ndcg_at_k(k):
        dcg_list = np.zeros(N, dtype=np.float32)
        for i in range(N):
            true_item = all_labels[i]
            topk_items = rank[i, :k]
            if true_item in topk_items:
                r = int(np.where(topk_items == true_item)[0][0])
                dcg_list[i] = 1.0 / np.log2(r + 2.0)
            else:
                dcg_list[i] = 0.0
        return float(dcg_list.mean())

    NDCG5  = compute_ndcg_at_k(5)
    NDCG10 = compute_ndcg_at_k(10)

    rr_list = np.zeros(N, dtype=np.float32)
    for i in range(N):
        true_item = all_labels[i]
        r_full = int(np.where(rank[i] == true_item)[0][0])
        rr_list[i] = 1.0 / (r_full + 1.0)
    MRR = float(rr_list.mean())

    return {
        "HR@1":   HR1,
        "HR@5":   HR5,
        "HR@10":  HR10,
        "NDCG@5":  NDCG5,
        "NDCG@10": NDCG10,
        "MRR":    MRR
    }

for epoch in tqdm(range(1, 1 + EPOCHS)):
    # --- (1) Train ---
    model_bert.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)              # (B, L, V)
        B, L, V = logits.shape
        loss = criterion(logits.view(-1, V), y.view(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    # --- (2) Validation ---
    model_bert.eval()
    val_loss = 0
    all_scores, all_labels = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model_bert(x)           # (B, L, V)
            B, L, V = logits.shape
            val_loss += criterion(logits.view(-1, V), y.view(-1)).item()

            # 평가 시: penultimate (=val_targets)만 예측
            for b in range(B):
                pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()  # 마지막 non-pad 위치
                scores = logits[b, pos, :].clone()

                # 이미 시퀀스에 등장한 아이템 제외
                input_token_ids = set(x[b].tolist())
                for t_id in input_token_ids:
                    if t_id != 0:
                        scores[t_id] = float('-inf')

                all_scores.append(scores.cpu().numpy())
                all_labels.append(y[b, pos].item())

    val_loss /= len(val_loader)
    # P@5, R@5, HR@5, F1@5, nDCG@5 등 계산:
    val_metrics = {}
    # (이하 val_metrics 계산 함수 호출 부분은 앞서 정의된 evaluate_ranking 이용)
    val_metrics = evaluate_simple_metrics(np.stack(all_scores), np.array(all_labels))

    # --- (3) Validation Loss 기준으로 모델 저장 ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch    = epoch
        save_path = os.path.join(SAVE_DIR, "best_bert4rec_leave2out.pt")
        torch.save(model_bert.state_dict(), save_path)

    # --- (4) wandb 로깅/출력 (필요 시) ---
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_HR@1": val_metrics['HR@1'],
        "val_HR@5": val_metrics['HR@5'],
        "val_HR@10": val_metrics['HR@10'],
        "val_NDCG@5": val_metrics['NDCG@5'],
        "val_NDCG@10": val_metrics['NDCG@10'],
        "val_MRR": val_metrics['MRR']
    })

print(f"▶ Best Model (Leave-2-Out) 저장 → Epoch {best_epoch} | Val Loss {best_val_loss:.4f}")

# ======================================================
# 7) Test 평가 (Leave-two-out)
#    - 저장된 best 모델 불러와서 penultimate가 아니라 “마지막” 토큰 평가
# ======================================================


# (1) 저장된 최적 모델 불러오기
model_bert.load_state_dict(torch.load(save_path))
model_bert.eval()

# (2) Test 데이터(leave-two-out의 “원본 전체 시퀀스”)에서 점수와 정답 수집
test_loss = 0
all_scores, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model_bert(x)           # (B, L, V)
        B, L, V = logits.shape
        test_loss += criterion(logits.view(-1, V), y.view(-1)).item()

        for b in range(B):
            # y[b]에서 -100이 아닌 위치 = “마지막 non-pad 위치” (원본 시퀀스의 마지막 아이템)
            pos = (y[b] != -100).nonzero(as_tuple=True)[0].item()
            scores = logits[b, pos, :].clone()  # (V,)

            # 이미 시퀀스에 등장한 아이템(excluding pad=0)은 점수 -inf 처리
            input_token_ids = set(x[b].tolist())
            for t_id in input_token_ids:
                if t_id != 0:
                    scores[t_id] = float('-inf')

            all_scores.append(scores.cpu().numpy())
            all_labels.append(y[b, pos].item())

test_loss /= len(test_loader)

# (3) Test 지표 계산
all_scores_np = np.stack(all_scores)
all_labels_np = np.array(all_labels, dtype=int)
test_metrics = evaluate_simple_metrics(all_scores_np, all_labels_np)

# (4) Test 결과 출력
print("\n===== Leave-2-Out Test 결과 =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"HR@1:   {test_metrics['HR@1']:.4f}")
print(f"HR@5:   {test_metrics['HR@5']:.4f}")
print(f"HR@10:  {test_metrics['HR@10']:.4f}")
print(f"NDCG@5: {test_metrics['NDCG@5']:.4f}")
print(f"NDCG@10:{test_metrics['NDCG@10']:.4f}")
print(f"MRR:    {test_metrics['MRR']:.4f}")


wandb: Currently logged in as: kwon04210 (listwiserank) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 100/100 [04:49<00:00,  2.89s/it]


▶ Best Model (Leave-2-Out) 저장 → Epoch 100 | Val Loss 3.1424

===== Leave-2-Out Test 결과 =====
Test Loss: 3.3357
HR@1:   0.2024
HR@5:   0.4911
HR@10:  0.6442
NDCG@5: 0.3503
NDCG@10:0.3995
MRR:    0.3407


In [8]:
import random
import torch


max_len       = config['max_seq_length']
mask_token_id = token2id['[MASK]']

# 예시 1개 시퀀스만 직접 지정 (길이가 5)
random_seqs = [
    ['Python', 'PyTorch','CERT_AI', 'CERT_IPE', 'CERT_DATA', 'TensorFlow', 'AWS'],
    ['HTMLCSS'],
    ['HTMLCSS', 'JS', 'CERT_IPE', 'Git', 'React', 'Redux'],
    ['HTMLCSS', 'JS', 'TYPE_Proj|ROLE_FE', 'TYPE_Club|ROLE_FE', 'TYPE_Junior|ROLE_FE', 'TYPE_Intern|ROLE_FE', 'TYPE_Proj|ROLE_UXUI'],
    []
]

model_bert.eval()

with torch.no_grad():
    for idx, seq in enumerate(random_seqs, 1):
        # 1) 토큰 → ID 변환 (토큰이 매핑에 없으면 에러가 나므로, 반드시 token2id 내에 존재하는 토큰만 사용)
        ids = [token2id[t] for t in seq]

        # 2) max_len-1 만큼만 뒤쪽을 남기기 (마지막에 [MASK] 자리를 하나 확보)
        ids = ids[-(max_len - 1):]  # e.g. max_len=20이면 max_len-1=19개까지만 남김

        # 3) 마지막에 [MASK] 토큰 추가
        masked_ids = ids + [mask_token_id]  # 길이가 <= max_len

        # 4) 남는 부분은 0 (padding)으로 채우기
        pad = [0] * (max_len - len(masked_ids))
        input_ids = pad + masked_ids      # 길이 = max_len

        # 5) 텐서로 변환 후 device로 이동
        inp = torch.tensor([input_ids], dtype=torch.long, device=device)  # shape=(1, max_len)

        # 6) 모델에 넣어서 로짓 계산
        logits = model_bert(inp)  # shape=(1, max_len, V) , V = n_items+1 (padding 제외)

        # 7) 마스크된 위치의 로짓 벡터만 추출 (맨 마지막 인덱스 = max_len-1)
        last_logits = logits[0, max_len - 1].clone()  # shape=(V,)

        # 8) 이미 시퀀스에 포함된 토큰 ID들은 후보에서 제외 (score=-inf)
        #    input_ids에는 padding(0), 실제 아이템 ID, [MASK] ID 등이 들어 있음
        for t_id in set(input_ids):
            if t_id != 0:           # padding(0)은 제외
                last_logits[t_id] = float('-inf')

        # 9) top-5 후보 ID 추출
        topk_ids = torch.topk(last_logits, k=5).indices.tolist()

        # 10) ID → 실제 토큰(아이템 이름)으로 변환
        topk_tokens = [id2token[i] for i in topk_ids]

        # 결과 출력
        print(f"\nTest#{idx}  입력시퀀스: {seq}")
        print(f"추천 Top-5 (중복 제외): {topk_tokens}")



Test#1  입력시퀀스: ['Python', 'PyTorch', 'CERT_AI', 'CERT_IPE', 'CERT_DATA', 'TensorFlow', 'AWS']
추천 Top-5 (중복 제외): ['TYPE_Research|ROLE_AI', 'TYPE_Proj|ROLE_AI', 'AWARD_OUTER', 'TYPE_Hackathon|ROLE_AI', 'TYPE_Club|ROLE_AI']

Test#2  입력시퀀스: ['HTMLCSS']
추천 Top-5 (중복 제외): ['JS', 'CERT_IPE', 'Git', 'React', 'Redux']

Test#3  입력시퀀스: ['HTMLCSS', 'JS', 'CERT_IPE', 'Git', 'React', 'Redux']
추천 Top-5 (중복 제외): ['TYPE_Proj|ROLE_FE', 'TYPE_Club|ROLE_FE', 'TYPE_Junior|ROLE_FE', 'TYPE_Intern|ROLE_FE', 'TYPE_Proj|ROLE_UXUI']

Test#4  입력시퀀스: ['HTMLCSS', 'JS', 'TYPE_Proj|ROLE_FE', 'TYPE_Club|ROLE_FE', 'TYPE_Junior|ROLE_FE', 'TYPE_Intern|ROLE_FE', 'TYPE_Proj|ROLE_UXUI']
추천 Top-5 (중복 제외): ['TYPE_Club|ROLE_UXUI', 'TYPE_Club|ROLE_DEVOPS', 'TYPE_Proj|ROLE_DEVOPS', 'TYPE_Hackathon|ROLE_FE', 'CERT_IPE']

Test#5  입력시퀀스: []
추천 Top-5 (중복 제외): ['Python', 'HTMLCSS', 'CERT_IPE', 'JS', 'SQL']


In [2]:
token2id

{'AWARD_OUTER': 1,
 'AWARD_UNIV': 2,
 'AWS': 3,
 'Angular': 4,
 'Ansible': 5,
 'ApacheSpark': 6,
 'Axios': 7,
 'Azure': 8,
 'AzureDevOps': 9,
 'Bash': 10,
 'Bootstrap': 11,
 'CERT_AI': 12,
 'CERT_AWS': 13,
 'CERT_DATA': 14,
 'CERT_IPE': 15,
 'CERT_OUTER': 16,
 'CUDA': 17,
 'Celery': 18,
 'CircleCI': 19,
 'Cypress': 20,
 'Django': 21,
 'Docker': 22,
 'DockerCompose': 23,
 'DynamoDB': 24,
 'ELK': 25,
 'Electron': 26,
 'Expo': 27,
 'Express': 28,
 'FAISS': 29,
 'FastAPI': 30,
 'Flask': 31,
 'Fluentd': 32,
 'GCP': 33,
 'Gin': 34,
 'Git': 35,
 'GitLabCI': 36,
 'Go': 37,
 'Grafana': 38,
 'HTMLCSS': 39,
 'Helm': 40,
 'JS': 41,
 'JWT': 42,
 'Java': 43,
 'JavaScript': 44,
 'Jenkins': 45,
 'Jest': 46,
 'Kafka': 47,
 'Keras': 48,
 'Kotlin': 49,
 'Kubernetes': 50,
 'Matplotlib': 51,
 'Milvus': 52,
 'MongoDB': 53,
 'MySQL': 54,
 'Next': 55,
 'NgRx': 56,
 'Node': 57,
 'NumPy': 58,
 'Numpy': 59,
 'Nuxt': 60,
 'OAuth2': 61,
 'Oracle': 62,
 'PHP': 63,
 'Pandas': 64,
 'PostgreSQL': 65,
 'PowerShell': 66